In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("penerimaan.csv")

# Pastikan kolom tanggal berformat datetime
df["datebayar"] = pd.to_datetime(df["datebayar"])

# Agregasi total penerimaan per hari/bulan
df_daily = df.groupby("datebayar")["nominal"].sum().reset_index()

# Atau Agregasi Bulanan (paling umum untuk prediksi penerimaan pajak)
df_monthly = (
    df.set_index("datebayar")
    .resample("MS")["nominal"]
    .sum()
    .reset_index()
)

In [ ]:
import numpy as np
import pandas as pd

# Library ML & Evaluasi
import joblib  # Untuk menyimpan model
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

# 1. Agregasi data bulanan
df["datebayar"] = pd.to_datetime(df["datebayar"])
df_monthly = (
    df.groupby(pd.Grouper(key="datebayar", freq="MS"))["nominal"]
    .sum()
    .reset_index()
)

# 2. Buat Fitur Waktu & Lag
df_monthly["bulan"] = df_monthly["datebayar"].dt.month
df_monthly["tahun"] = df_monthly["datebayar"].dt.year
df_monthly["lag_1"] = df_monthly["nominal"].shift(1)
df_monthly["lag_12"] = df_monthly["nominal"].shift(12)  # Musiman 1 tahun lalu

# Hapus baris kosong akibat shift/lag
df_clean = df_monthly.dropna().copy()

# 3. Pisahkan Fitur (X) dan Target (y)
X = df_clean[["bulan", "tahun", "lag_1", "lag_12"]]
y = df_clean["nominal"]

# Split Data (misal 80% Latih, 20% Uji)
split_idx = int(len(df_clean) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 4. Melatih Model Machine Learning
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Prediksi & Evaluasi Statistik
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

print(f"--- EVALUASI STATISTIK MODEL ---")
print(f"R-squared (R2) : {r2:.4f}")
print(f"MAE            : Rp {mae:,.2f}")
print(f"RMSE           : Rp {rmse:,.2f}")
print(f"MAPE           : {mape:.2f}%")

# 6. Simpan Model untuk Aplikasi (.exe)
joblib.dump(model, "model_pajak.pkl")
print("Model berhasil disimpan ke 'model_pajak.pkl'")

In [ ]:
import joblib
import pandas as pd

# 1. Muat dataset dan model yang sudah disimpan
df = pd.read_csv("penerimaan.csv")
model = joblib.load("model_pajak.pkl")

# 2. Pastikan format tanggal sesuai
df["datebayar"] = pd.to_datetime(df["datebayar"])

# 3. Agregasi total bulanan
df_monthly = (
    df.groupby(pd.Grouper(key="datebayar", freq="MS"))["nominal"]
    .sum()
    .reset_index()
)

# 4. Ambil data historis yang dibutuhkan untuk prediksi Januari 2026
# lag_1  = Desember 2025
# lag_12 = Januari 2025

val_des_2025 = df_monthly[df_monthly["datebayar"] == "2025-12-01"][
    "nominal"
].values[0]
val_jan_2025 = df_monthly[df_monthly["datebayar"] == "2025-01-01"][
    "nominal"
].values[0]

# 5. Siapkan data input untuk model [[bulan, tahun, lag_1, lag_12]]
input_data = [[1, 2026, val_des_2025, val_jan_2025]]

# 6. Jalankan Prediksi
prediksi_jan_2026 = model.predict(input_data)[0]

print(f"--- HASIL PREDIKSI PENERIMAAN PAJAK ---")
print(f"Target Periode      : Januari 2026")
print(f"Penerimaan Des 2025 : Rp {val_des_2025:,.2f}")
print(f"Penerimaan Jan 2025 : Rp {val_jan_2025:,.2f}")
print(
    f"----------------------------------------"
)
print(f"PREDIKSI JANUARI 2026: Rp {prediksi_jan_2026:,.2f}")